In [11]:
import open_clip
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')

c:\Learning\Labs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Learning\Labs\venv\lib\site-packages\huggingface_hub\file_download.py:121: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\madhu\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, 

In [24]:
import torch
import open_clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = open_clip.get_tokenizer('ViT-B-32')

image = preprocess(Image.open("test.jpg")).unsqueeze(0).to(device)
text = tokenizer(["a man at beach sitting", "a man at beach diving"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)

print("Image features:", image_features)
print("Text features:", text_features)

Image features: tensor([[-1.6421e-01,  1.2765e-01,  1.7337e-01,  2.3001e-01,  4.8105e-01,
         -4.0254e-03, -1.9595e-01, -1.5289e-02,  3.0144e-01, -9.1757e-02,
          4.3155e-02, -2.7956e-01, -6.5352e-02,  4.9623e-02, -2.5573e-01,
          6.4865e-02, -2.5828e-02, -1.3281e-01,  4.3938e-01, -8.2537e-01,
         -1.2299e+00,  9.4261e-02,  4.2761e-01,  4.2382e-01, -2.9760e-01,
          4.5100e-01, -1.9786e-01,  1.8974e-02,  5.4217e-02, -2.3189e-01,
         -1.4500e-01,  3.0748e-01,  2.7891e-01, -4.8976e-02, -1.6231e-01,
         -6.6740e-02,  3.6242e-01, -7.6054e-03, -2.1225e-02,  4.1495e-01,
         -1.3269e-01, -1.3112e-01, -2.4012e-01, -9.5677e-02, -6.2154e-03,
         -1.0164e+00,  8.7454e-01, -2.8933e-01, -3.7534e-01,  2.2253e-01,
         -5.2021e-01,  1.9805e-01,  1.1298e-01,  1.2568e-01, -9.4489e-02,
          1.8902e-01, -1.8982e-01, -5.6720e-02,  1.7866e-01,  5.3229e-01,
          8.7672e-01,  2.9875e-01,  1.1955e-01, -5.5821e-02, -3.9140e-02,
         -1.3153e-01, 

In [25]:
similarity = (image_features @ text_features.T).softmax(dim=-1)
print(similarity)

tensor([[0.0384, 0.9616]])


In [ ]:
import os
image_embs = [] 

image_folder = "../datasets/flickr8k/Images/"

for filename in os.listdir(image_folder):
    path = os.path.join(image_folder, filename)

    img = preprocess(Image.open(path)).unsqueeze(0).to(device)    
    with torch.no_grad():
        emb = model.encode_image(img)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        image_embs.append(emb)
    
    show_progress = len(image_embs) % 100 == 0
    if show_progress:
        print(f"Processed {len(image_embs)} images")

image_embs = torch.cat(image_embs)


Processed 100 images
Processed 200 images
Processed 300 images
Processed 400 images
